# Baselines: The Bar the GDBN Has to Beat

Predict each series' **next-day return** three simple ways, and score them on the **validation set (2021–2022)**:

| Baseline | Prediction for tomorrow |
|---|---|
| **zero** | 0 (no change) |
| **hist_mean** | the series' average daily return in training (2006–2020) |
| **ar1** | from only its own previous day (a regression fit on training) |

**Metrics** (per series, then averaged over the 35 return series):
- **mse:** mean squared error (lower is better).
- **r2_os:** out-of-sample $R^2$ vs the historical mean, $1 - \text{MSE}/\text{MSE}_{\text{hist\_mean}}$. Positive = beats the historical mean. In finance, even +0.5% is considered meaningful for daily returns.
- **direction:** share of days the predicted sign is right (a prediction of 0 counts as "up").
- **coverage90:** each baseline also gives an uncertainty range (± 1.645 × its training error). A well-calibrated range contains the actual return **90%** of the time.

The **test set (2023 onward) is not used here.**

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from gdbn_utils import load_data, make_pairs, baseline_predictions, evaluate, summarize, dm_report, RESULTS

pd.set_option("display.precision", 4)
wide, split, info = load_data()
RET = info["RET"]
X, Y, labels = make_pairs(wide, split)
print("pairs per split:", labels.value_counts().reindex(["train", "validation", "test"]).to_dict())

## Predictions and validation scores

In [ ]:
preds, sigmas = baseline_predictions(X, Y, labels, RET)
val = labels == "validation"
y_val = Y.loc[val, RET]

bench_mse = ((y_val - preds["hist_mean"].loc[val]) ** 2).mean()
per_series = {name: evaluate(y_val, p.loc[val], sigmas[name], bench_mse) for name, p in preds.items()}
table = pd.DataFrame({name: summarize(df) for name, df in per_series.items()}).T
table["mse"] *= 1e4  # in (percent)^2 per day, easier to read
table = table.rename(columns={"mse": "mse (%^2)"})
table

**Reading the table:**
- The **mse** values are nearly identical: next-day returns are mostly unpredictable, so all baselines are about equally (in)accurate.
- **direction** for **zero** is simply "always up" (a prediction of 0 counts as up), so it equals the share of up days in 2021–2022. For **hist_mean**, it is "up" for series whose training average was positive. Anything near 50% is no better than a coin flip.
- **coverage90** below 90% means validation was **more volatile than training**: the uncertainty ranges learned in 2006–2020 were too narrow.

## Per-series view: does AR(1) help anywhere?

In [ ]:
r2 = per_series["ar1"]["r2_os"].sort_values()
fig, ax = plt.subplots(figsize=(10, 4))
r2.mul(100).plot.bar(ax=ax, color=np.where(r2 > 0, "#55a868", "#c44e52"))
ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("out-of-sample R² vs hist mean (%)"); ax.set_title("AR(1) on validation (2021–2022): green = beats historical mean")
ax.set_xticklabels([c.replace("_ret", "") for c in r2.index], rotation=90, fontsize=8)
plt.tight_layout(); plt.show()
print(f"AR(1) beats the historical mean on {(r2 > 0).sum()} of {len(r2)} series")

## Is AR(1) significantly different from the historical mean?

**Diebold–Mariano test:** compares two forecasts' squared errors day by day. A small p-value (< 0.05) means the difference is real, not luck, and the output says **which** forecast is better. (A significant result can also mean a model is significantly *worse*.) This is the kind of test the RSR paper's results did not pass.

In [ ]:
err = {name: y_val - p.loc[val] for name, p in preds.items()}
for a, b in [("ar1", "hist_mean"), ("zero", "hist_mean")]:
    print(dm_report(a, b, err[a], err[b]))

## Save for the GDBN notebook

In [ ]:
RESULTS.mkdir(parents=True, exist_ok=True)
table.to_csv(RESULTS / "baselines_validation.csv")
pd.concat({name: df for name, df in per_series.items()}, names=["baseline", "series"]).to_csv(RESULTS / "baselines_validation_per_series.csv")
print("Saved to", RESULTS)